In [ ]:
"""
QC Script: Counts and lists 'bad' windows for metrics used in downstream analysis.

Summary of logic:
-----------------
We slice each pose-tracking CSV file into fixed-size windows.
We mark keypoints as "bad" if they have too long of a consecutive missing-data gap.
We mark metrics as "bad" if ANY of their constituent keypoints are bad.
We produce:
    1) Keypoint-level bad window counts
    2) Metric-level bad window counts
    3) Metric-level *indices* of bad windows for inspection

Definitions:
------------
- A keypoint is considered present in a frame if:
    prob_i >= CONFIDENCE_THRESHOLD  AND
    both x_i, y_i are finite (non-NaN)
- In a given window, a keypoint is "bad" if:
    longest consecutive run of missing frames > MAX_INTERP
- A metric is "bad" in a window if:
    ANY keypoint used by that metric is bad in that window

Config parameters to adjust:
----------------------------
- INPUT_DIR, OUTPUT_DIR
- WINDOW_SIZE (frames), OVERLAP (fraction)
- CONFIDENCE_THRESHOLD, MAX_INTERP (max allowed gap)
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

# --------- Config ---------
INPUT_DIR = "data/pose/experimental_pose" # change to experimental or baseline as needed
OUTPUT_DIR = "data/qc_outputs_exp"
WINDOW_SIZE = 1800               # Number of frames per QC window
OVERLAP = 0.0                    # Fractional overlap between consecutive windows (0=no overlap)
CONFIDENCE_THRESHOLD = 0.3       # Min probability for a keypoint to be "present"
MAX_INTERP = 60                  # Max allowed consecutive missing frames before keypoint considered "bad"
# --------------------------

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Mapping of metrics to the keypoint indices they depend on
METRIC_KPS = {
    "eyes":            [37, 38, 40, 41, 43, 44, 46, 47],  # blink-related kps
    "head_rotation":   [36, 45],
    "mouth_dist":      [62, 66],
    "pupils_combined": [68, 69],
    "center_face":     list(range(27, 36)),  # nose/center face landmarks
}

# Flat sorted list of all relevant keypoints used by ANY metric
RELEVANT_KPS = sorted({kp for kps in METRIC_KPS.values() for kp in kps})

# ---------------- Utility Functions ---------------- #

def load_csv(fp: str) -> pd.DataFrame:
    """Reads a pose CSV file into a DataFrame."""
    return pd.read_csv(fp)

def window_ranges(n_rows: int, window_size: int, overlap: float):
    """
    Given the number of rows in the file, returns a list of (start, end) indices for windows.
    Overlap is fractional; step size = window_size * (1 - overlap).
    If the file is shorter than one window, return [].
    """
    if n_rows < window_size:
        return []
    step = max(1, int(round(window_size * (1 - overlap))))  # Ensure step >= 1 frame
    return [(s, s + window_size) for s in range(0, n_rows - window_size + 1, step)]

def _max_true_run_length(b: pd.Series) -> int:
    """
    Returns the length of the longest consecutive run of True values in a boolean Series.
    Used to detect the longest gap of missing keypoint data.
    """
    run = max_run = 0
    arr = b.to_numpy()
    for v in arr:
        if v:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def kp_missing_series(df_w: pd.DataFrame, i: int) -> pd.Series:
    """
    Returns a boolean Series where True = keypoint i is missing in that frame.
    Missing means:
        - Probability < CONFIDENCE_THRESHOLD
        OR
        - x or y coordinate is NaN
    If the columns for this keypoint are absent in the file, treat it as fully missing.
    """
    xcol, ycol, pcol = f"x{i}", f"y{i}", f"prob{i}"
    if (xcol not in df_w.columns) or (ycol not in df_w.columns) or (pcol not in df_w.columns):
        return pd.Series(True, index=df_w.index)  # Fully missing
    good = (df_w[pcol] >= CONFIDENCE_THRESHOLD) & df_w[xcol].notna() & df_w[ycol].notna()
    return ~good  # invert: True means missing

# ---------------- Core Analysis ---------------- #

def analyze_file(fp: str):
    """
    Process a single CSV file:
        - Slice into windows
        - Detect 'bad' keypoints per window
        - Aggregate to metric-level badness
        - Return three DataFrames: keypoint summary, metric summary, metric bad window indices
    """
    df = load_csv(fp)
    base = os.path.basename(fp)
    wranges = window_ranges(len(df), WINDOW_SIZE, OVERLAP)
    total_windows = len(wranges)

    # If file is too short for even one window, return empty summaries
    if total_windows == 0:
        kp_rows = [{
            "file": base, "keypoint": i, "bad_windows": 0,
            "total_windows": 0, "pct_bad": np.nan
        } for i in RELEVANT_KPS]
        met_rows = [{
            "file": base, "metric": m, "bad_windows": 0,
            "total_windows": 0, "pct_bad": np.nan
        } for m in METRIC_KPS.keys()]
        met_idx_rows = []  # no bad window indices
        return pd.DataFrame(kp_rows), pd.DataFrame(met_rows), pd.DataFrame(met_idx_rows)

    # Tracking dictionaries
    kp_bad_counts = {i: 0 for i in RELEVANT_KPS}        # counts of bad windows per keypoint
    metric_bad_counts = {m: 0 for m in METRIC_KPS}      # counts of bad windows per metric
    metric_bad_indices = {m: [] for m in METRIC_KPS}    # stores (win_idx, start_frame, end_frame)

    # Iterate over each window
    for win_idx, (s, e) in enumerate(wranges):
        df_w = df.iloc[s:e].reset_index(drop=True)

        # --- Keypoint-level QC ---
        kp_bad_in_this_window = {}
        for i in RELEVANT_KPS:
            missing = kp_missing_series(df_w, i)
            longest_gap = _max_true_run_length(missing)
            is_bad = (longest_gap > MAX_INTERP)  # too long of a missing-data gap
            kp_bad_in_this_window[i] = is_bad
            if is_bad:
                kp_bad_counts[i] += 1

        # --- Metric-level QC ---
        for m, kps in METRIC_KPS.items():
            # Metric is bad if ANY constituent keypoint is bad
            if any(kp_bad_in_this_window.get(i, True) for i in kps):
                metric_bad_counts[m] += 1
                metric_bad_indices[m].append((win_idx, s, e))

    # --- Build per-keypoint summary table ---
    kp_rows = []
    for i in RELEVANT_KPS:
        bw = kp_bad_counts[i]
        kp_rows.append({
            "file": base,
            "keypoint": i,
            "bad_windows": bw,
            "total_windows": total_windows,
            "pct_bad": bw / total_windows if total_windows > 0 else np.nan
        })

    # --- Build per-metric summary table ---
    met_rows = []
    for m in METRIC_KPS:
        bw = metric_bad_counts[m]
        met_rows.append({
            "file": base,
            "metric": m,
            "bad_windows": bw,
            "total_windows": total_windows,
            "pct_bad": bw / total_windows if total_windows > 0 else np.nan
        })

    # --- Build per-metric bad window indices table ---
    met_idx_rows = []
    for m, entries in metric_bad_indices.items():
        for (win_idx, s, e) in entries:
            met_idx_rows.append({
                "file": base,
                "metric": m,
                "window_index": win_idx,
                "start_frame": s,
                "end_frame_exclusive": e
            })

    return pd.DataFrame(kp_rows), pd.DataFrame(met_rows), pd.DataFrame(met_idx_rows)

# ---------------- Main Driver ---------------- #

def main():
    """Scan all CSV files in INPUT_DIR and write QC summaries to OUTPUT_DIR."""
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".csv")]
    all_kp = []
    all_met = []
    all_met_idx = []

    for f in tqdm(files, desc="QC scanning"):
        fp = os.path.join(INPUT_DIR, f)
        try:
            kp_df, met_df, met_idx_df = analyze_file(fp)
        except Exception as e:
            # Catch errors and log them in output tables
            kp_df = pd.DataFrame([{
                "file": f, "keypoint": None, "bad_windows": None,
                "total_windows": None, "pct_bad": None, "error": str(e)
            }])
            met_df = pd.DataFrame([{
                "file": f, "metric": None, "bad_windows": None,
                "total_windows": None, "pct_bad": None, "error": str(e)
            }])
            met_idx_df = pd.DataFrame([{
                "file": f, "metric": None, "window_index": None,
                "start_frame": None, "end_frame_exclusive": None, "error": str(e)
            }])

        all_kp.append(kp_df)
        all_met.append(met_df)
        all_met_idx.append(met_idx_df)

    # Merge results from all files
    out_kp = pd.concat(all_kp, ignore_index=True)
    out_met = pd.concat(all_met, ignore_index=True)
    out_met_idx = pd.concat(all_met_idx, ignore_index=True)

    # Write outputs
    out_kp_path = os.path.join(OUTPUT_DIR, "keypoint_bad_windows.csv")
    out_met_path = os.path.join(OUTPUT_DIR, "metric_bad_windows.csv")
    out_met_idx_path = os.path.join(OUTPUT_DIR, "metric_bad_window_indices.csv")

    out_kp.to_csv(out_kp_path, index=False)
    out_met.to_csv(out_met_path, index=False)
    out_met_idx.to_csv(out_met_idx_path, index=False)

    print("Wrote:")
    print(f"  {out_kp_path}")
    print(f"  {out_met_path}")
    print(f"  {out_met_idx_path}")

if __name__ == "__main__":
    main()


QC scanning: 100%|██████████| 215/215 [00:14<00:00, 14.77it/s]

Wrote:
  data/qc_outputs_bsl/keypoint_bad_windows.csv
  data/qc_outputs_bsl/metric_bad_windows.csv
  data/qc_outputs_bsl/metric_bad_window_indices.csv


In [ ]:
import pandas as pd

def summarize_bad_windows(path):
    df = pd.read_csv(path)
    total_bad = df['bad_windows'].sum()
    total_windows = df['total_windows'].sum()
    pct = total_bad / total_windows * 100 if total_windows > 0 else float('nan')
    return total_bad, total_windows, pct

bsl_bad, bsl_total, bsl_pct = summarize_bad_windows("data/qc_outputs_bsl/keypoint_bad_windows.csv")
exp_bad, exp_total, exp_pct = summarize_bad_windows("data/qc_outputs_exp/keypoint_bad_windows.csv")

print(f"Baseline: {bsl_bad}/{bsl_total} windows bad ({bsl_pct:.2f}%)")
print(f"Experimental: {exp_bad}/{exp_total} windows bad ({exp_pct:.2f}%)")
